#  Clasificación de Objetos Astronómicos 

## --- Carga y Preparación de Datos ---
 Se imortan las librerias a utilizar y se carga el dataset, crea nuevas features (colores) y separa en entrenamiento y prueba.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score


file_path = '../data/raw/Skyserver_SQL6_1_2025 5_01_52 PM.csv'
df = pd.read_csv(file_path, header=1)
print("Total de datos:", len(df))
print(df.head())

df['u-g'] = df['u'] - df['g']
df['g-r'] = df['g'] - df['r']
df['r-i'] = df['r'] - df['i']
df['i-z'] = df['i'] - df['z']

features = ['u', 'g', 'r', 'i', 'z', 'u-g', 'g-r', 'r-i', 'i-z', 'redshift']
X = df[features]
le = LabelEncoder()
y = le.fit_transform(df['class'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## --- Funciones Auxiliares ---
 Se definen funciones para calcular RMSE, evaluar modelos con PCA y GridSearchCV, y mostrar resultados.

In [ ]:

def calcular_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mostrar_resultados(modelo_nombre, y_test, y_pred, modelo_entrenado, X_train_scaled):
    print(f"\n- Reporte de Clasificación - {modelo_nombre}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

## --- Evaluación de Modelos ---
Se evaluan 3 modelos y se recorre cada uno con GridSearchCV y PCA para encontrar la mejor combinación de cada uno.


In [ ]:

resultados = {}
modelos = [
    ('LogisticRegression', LogisticRegression(max_iter=1000), {'C': [0.1, 1, 10]}),
    ('KNN', KNeighborsClassifier(), {'n_neighbors': [3, 5, 7]}),
    ('RandomForest', RandomForestClassifier(), {'n_estimators': [50, 100]})
]

for nombre, modelo, param_grid in modelos:
    print(f"Evaluando {nombre}...")
    mejor_rmse = float('inf')
    mejor_resultado = None

    for n_pca in list(range(2, X_train.shape[1] + 1)) + [None]:
        if n_pca is None:
            # Sin PCA
            pasos = [
                ('scaler', StandardScaler()),
                ('clf', modelo)
            ]
        else:
            # Con PCA
            pasos = [
                ('scaler', StandardScaler()),
                ('pca', PCA(n_components=n_pca)),
                ('clf', modelo)
            ]
        
        pipe = Pipeline(pasos)
        
        param_grid_prep = {'clf__' + k: v for k, v in param_grid.items()}
        grid = GridSearchCV(pipe, param_grid_prep, cv=5, scoring='accuracy', n_jobs=-1)
        grid.fit(X_train, y_train)
        y_pred = grid.predict(X_test)
        rmse = calcular_rmse(y_test, y_pred)

        if rmse < mejor_rmse:
            mejor_resultado = {
                'mejor_modelo': grid.best_estimator_,
                'mejores_parametros': grid.best_params_,
                'rmse': rmse,
                'n_pca': n_pca,
                'reporte': classification_report(y_test, y_pred, target_names=le.classes_, output_dict=False)
            }
            mejor_rmse = rmse


    resultados[nombre] = mejor_resultado
    mostrar_resultados(nombre, y_test, grid.predict(X_test), mejor_resultado['mejor_modelo'], X_train)

## --- Comparación Final de Modelos ---
 Se construye un resumen con los RMSEs y se visualiza la comparación gráfica.


In [ ]:
df_comparacion = pd.DataFrame([
    {
        'Modelo': k,
        'Accuracy': accuracy_score(y_test, v['mejor_modelo'].predict(X_test)),
        'Macro F1-Score': f1_score(y_test, v['mejor_modelo'].predict(X_test), average='macro'),
        'n_PCA': v['n_pca']
    } for k, v in resultados.items()
]).sort_values(by='Macro F1-Score', ascending=False)

print("\n📊 Comparación de Modelos por Accuracy y Macro F1-Score:")
print(df_comparacion)

# Gráfico de comparación por Accuracy
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.barplot(data=df_comparacion, x='Modelo', y='Accuracy')
plt.title('Comparación de Modelos por Accuracy')
plt.xticks(rotation=45)

# Gráfico de comparación por Macro F1-Score
plt.subplot(1, 2, 2)
sns.barplot(data=df_comparacion, x='Modelo', y='Macro F1-Score')
plt.title('Comparación de Modelos por Macro F1-Score')


### Gráficos Comparativos

Los gráficos presentan el rendimiento de los tres modelos evaluados —**Random Forest**, **Logistic Regression** y **KNN**— usando dos métricas:

- **Accuracy**: Indica qué proporción de las predicciones fueron correctas sobre el total de observaciones.  
- **Macro F1-Score**: Mide el equilibrio entre precisión y exhaustividad para cada clase, y luego promedia esos valores. Como tenemos a las clases desbalanciadas, de esta forma se tratan por igual.

#### Comparacoion de resultados:

- **Random Forest** obtuvo el mejor rendimiento en ambas métricas, con una **accuracy del 99.05%** y un **macro F1-score de 0.982**.
- **Logistic Regression** también mostró buen desempeño, ligeramente inferior con un F1 de **0.963**.
- **KNN** fue el modelo con menor rendimiento, aunque aún mantiene valores bastante altos, con una accuracy de **96.55%**.

#### Conclusión:

Tanto en precisión global como en capacidad para clasificar correctamente todas las clases de forma equilibrada, **Random Forest** se destaca como el mejor modelo para este problema.


## Análisis del Modelo Seleccionado

Luego de comparar el rendimiento de los tres modelos candidatos, seleccionamos el modelo con mejor desempeño general siendo el modelo de **Random Forest**.

Analizemos más detalladamente sobre este modelo, presentando visualizaciones para comprender mejor cómo clasifica los datos:

- Matriz de Confusión
- Curva ROC por clase
- Distribución de componentes principales (PCA)
- Importancia de variables




In [ ]:
mejor_modelo = df_comparacion.iloc[0]

if mejor_modelo['Modelo'] == "RandomForest":
    from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
    from sklearn.preprocessing import label_binarize
    from IPython.display import Markdown, display

    rf_resultado = resultados["RandomForest"]
    modelo_rf = rf_resultado['mejor_modelo']
    y_pred = modelo_rf.predict(X_test)
    y_pred_prob = modelo_rf.predict_proba(X_test)

    print("\n📌 Reporte de Clasificación - Random Forest")
    print(rf_resultado['reporte'])

    ConfusionMatrixDisplay.from_estimator(modelo_rf, X_test, y_test, display_labels=le.classes_, cmap='Blues')
    plt.title("Matriz de Confusión - Random Forest")
    plt.tight_layout()
    plt.show()

    importancias = modelo_rf.named_steps['clf'].feature_importances_
    variables = X.columns
    sns.barplot(x=importancias, y=variables)
    plt.title("Importancia de Variables - Random Forest")
    plt.tight_layout()
    plt.show()

    y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(len(le.classes_)):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(8,6))
    for i, label in enumerate(le.classes_):
        plt.plot(fpr[i], tpr[i], label=f"{label} (AUC = {roc_auc[i]:.2f})")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title("Curvas ROC por Clase - Random Forest")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend(loc="lower right")
    plt.grid()
    plt.tight_layout()
    plt.show()

    X_scaled = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2)
    X_proj = pca.fit_transform(X_scaled)

    plt.figure(figsize=(8,6))
    sns.scatterplot(x=X_proj[:, 0], y=X_proj[:, 1], hue=le.inverse_transform(y), alpha=0.5, palette='Set2')
    plt.title("PCA 2D - Clases Reales")
    plt.tight_layout()
    plt.show()

    y_pred_all = modelo_rf.predict(X_scaled)
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=X_proj[:, 0], y=X_proj[:, 1], hue=le.inverse_transform(y_pred_all), alpha=0.5, palette='Set1')
    plt.title("PCA 2D - Predicción del Modelo Random Forest")
    plt.tight_layout()
    plt.show()

## Matriz de Confusión

Este gráfico vemos el número de predicciones correctas e incorrectas por clase. Podemos ver que
La clase GALAXY fue identificada casi perfectamente con 1 error como QSO y 2 como STAR.
STAR no tuvo ningún error de clasificación.
QSO, que suele ser una clase más difícil, fue correctamente identificada en 185 casos de 201, con 16 clasificados erróneamente como GALAXY.


## Importancia de Variables

La importancia de las variables muestra qué características fueron más relevantes para el modelo.
Redshift fue la variable más influyente con diferencia.
Las combinaciones de bandas como r-i, g-r e i-z también aportan información relevante.
Las bandas individuales (u, g, r, i, z) tienen un peso menor.
Sugiriendo que el modelo se apoya principalmente en la información espectral y el corrimiento al rojo para clasificar objetos.

## Curvas ROC por Clase

Este gráfico evalúa la capacidad del modelo para distinguir entre clases:
Las áreas bajo la curva (AUC) son casi perfectas:
GALAXY y STAR con un AUC = 1.00
QSO con AUC = 0.99

La cercanía al vértice superior izquierdo del gráfico indica que el modelo tiene una excelente tasa de verdaderos positivos y baja tasa de falsos positivos para todas las clases.

## Visualización PCA en 2D 
El gráfico muestra una proyección 2D del dataset usando Análisis de Componentes Principales (PCA), coloreada según la clase real:

GALAXY, STAR y QSO muestran regiones relativamente bien definidas en el espacio reducido.

Algunas superposiciones existen (en especial QSO con STAR), lo que justifica pequeños errores en la clasificación.

Esto refuerza que el problema no es trivial, pero el modelo logra captar las estructuras principales de las clases.